# 🚀 Project 1 — Synthetic Text Generation

## 📌 Objective

The goal of this project is to generate synthetic customer support queries using a Large Language Model (LLM). This synthetic data is later used to build and evaluate downstream NLP systems such as classification models and retrieval-based systems.

---

## 🧠 Motivation

Real-world datasets are often:

* limited in size
* imbalanced across classes
* difficult to obtain due to privacy constraints

Synthetic data generation helps to:

* simulate realistic user queries
* create balanced datasets
* accelerate experimentation in NLP pipelines

---

## ⚙️ Approach

We use a lightweight instruction-tuned language model:

```text
microsoft/Phi-4-mini-instruct
```

### Key steps:

1. Define intent categories:

   * order_status
   * cancel_request
   * refund_request
   * complaint
   * product_question

2. Design prompts for each category

3. Generate multiple variations of user queries per category

4. Store results in structured format (CSV / JSON)

---

## 🧾 Example Generated Data

| Text                                   | Label          |
| -------------------------------------- | -------------- |
| I need to check the status of my order | order_status   |
| Can I cancel my order?                 | cancel_request |
| I want a refund for my purchase        | refund_request |

---

## 🛠️ Technical Details

* Model: `Phi-4-mini-instruct`
* Tokenization handled via HuggingFace `AutoTokenizer`
* Quantization: 4-bit using BitsAndBytes (for efficient local inference)
* Generation parameters:

  * temperature (controls diversity)
  * max tokens
  * sampling strategies

---

## 📦 Output

Generated datasets:

```text
data/synthetic/
├── synthetic_clean_300.csv
├── synthetic_clean_300.json
├── logs.csv
```

These datasets are used in:

👉 **Project 2 — RAG System (LangChain)**

---

## ⚠️ Limitations

The generated data represents:

* user intents / queries

It does NOT contain:

* factual knowledge
* policies
* procedural instructions

This limitation impacts downstream tasks such as:

* grounded question answering in RAG systems

---

## 🎯 Key Insight

> Synthetic data is useful for modeling user behavior, but not sufficient for knowledge-based QA systems.

---

## 🚀 Next Steps

* Use this dataset for:

  * text classification
  * semantic retrieval (RAG baseline)

* Build a knowledge-based dataset for:

  * grounded RAG QA systems


In [1]:
!git clone https://github.com/DeepuLIN/llm-text-learning-lab.git
%cd llm-text-learning-lab

!pip install -q --upgrade transformers accelerate bitsandbytes datasets

fatal: destination path 'llm-text-learning-lab' already exists and is not an empty directory.
/content/llm-text-learning-lab


In [2]:
# Core
import os
import time
import json
import re
import ast

# Data
import pandas as pd

# Torch
import torch

# Hugging Face
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

# Colab
from google.colab import userdata

In [3]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [4]:
MODEL_NAME = "microsoft/Phi-4-mini-instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quant_config
)

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [5]:
LABELS = [
    "order_status",
    "cancel_request",
    "refund_request",
    "product_question",
    "complaint"
]

In [6]:
def build_prompt():
    return f"""
You are generating synthetic data for intent classification.

Generate 10 customer support queries.

Allowed labels:
- order_status
- cancel_request
- refund_request
- product_question
- complaint

Examples:
Text: "Where is my order?"
Label: order_status

Text: "I want to cancel my subscription"
Label: cancel_request

Text: "Can I get a refund for this item?"
Label: refund_request

Text: "Does this product come in blue?"
Label: product_question

Text: "This item arrived damaged and I am unhappy"
Label: complaint

Return ONLY a JSON array.
Each item must have keys: text, label.
Do not repeat the instructions.
Do not include markdown.
Do not include explanations.
Start directly with [ and end with ].
"""

In [7]:
def generate_batch(prompt):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return response

In [8]:
def extract_last_json_array(text):
    matches = re.findall(r"\[\s*.*?\s*\]", text, re.DOTALL)
    if not matches:
        raise ValueError("No JSON array found")

    json_str = matches[-1].strip()

    try:
        return json.loads(json_str)
    except:
        return ast.literal_eval(json_str)

In [9]:
VALID_LABELS = set(LABELS)

def clean_batch(batch):
    cleaned = []
    for item in batch:
        text = str(item.get("text", "")).strip()
        label = str(item.get("label", "")).strip().lower()

        if not text:
            continue
        if label not in VALID_LABELS:
            continue

        cleaned.append({
            "text": text,
            "label": label,
            "source": "synthetic"
        })
    return cleaned

In [10]:
def generate_n_samples(target_samples=300, batch_size=10):
    all_rows = []
    batch_logs = []

    n_batches = target_samples // batch_size

    for i in range(n_batches):
        print(f"Batch {i+1}/{n_batches}")
        try:
            prompt = build_prompt()
            output = generate_batch(prompt)
            batch = extract_last_json_array(output)
            batch = clean_batch(batch)

            all_rows.extend(batch)

            batch_logs.append({
                "batch_id": i+1,
                "status": "success",
                "rows": len(batch)
            })

        except Exception as e:
            batch_logs.append({
                "batch_id": i+1,
                "status": f"failed: {e}",
                "rows": 0
            })

        time.sleep(1)

    df = pd.DataFrame(all_rows)
    logs_df = pd.DataFrame(batch_logs)

    return df, logs_df

In [11]:
df_raw, logs_df = generate_n_samples(300)

Batch 1/30
Batch 2/30
Batch 3/30
Batch 4/30
Batch 5/30
Batch 6/30
Batch 7/30
Batch 8/30
Batch 9/30
Batch 10/30
Batch 11/30
Batch 12/30
Batch 13/30
Batch 14/30
Batch 15/30
Batch 16/30
Batch 17/30
Batch 18/30
Batch 19/30
Batch 20/30
Batch 21/30
Batch 22/30
Batch 23/30
Batch 24/30
Batch 25/30
Batch 26/30
Batch 27/30
Batch 28/30
Batch 29/30
Batch 30/30


In [12]:
df = df_raw.copy()

df["text"] = df["text"].str.strip()
df["label"] = df["label"].str.lower()

df = df.dropna()
df = df[df["text"] != ""]
df = df[df["label"].isin(LABELS)]

before = len(df)

df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

print("Before:", before)
print("After:", len(df))
print(df["label"].value_counts())

Before: 280
After: 260
label
refund_request      60
order_status        52
product_question    50
complaint           50
cancel_request      48
Name: count, dtype: int64


In [13]:
os.makedirs("data/synthetic", exist_ok=True)

df.to_csv("data/synthetic/synthetic_clean_300.csv", index=False)
logs_df.to_csv("data/synthetic/logs.csv", index=False)

with open("data/synthetic/synthetic_clean_300.json", "w") as f:
    json.dump(df.to_dict(orient="records"), f, indent=2)

In [14]:
!zip -r synthetic_data.zip data/synthetic

  adding: data/synthetic/ (stored 0%)
  adding: data/synthetic/logs.csv (deflated 70%)
  adding: data/synthetic/synthetic_clean_300.csv (deflated 82%)
  adding: data/synthetic/synthetic_clean_300.json (deflated 89%)


In [15]:
from google.colab import files
files.download("synthetic_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>